In [2]:
# 07b – RoBERTa Base with Data Augmentation (Inference)

import os
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizer, RobertaModel
from tqdm.auto import tqdm
import kagglehub

# -------------------------
# 0. Config
# -------------------------
LABELS = ['anger','fear','joy','sadness','surprise']
MAX_LEN = 80
BATCH_SIZE = 16
THRESHOLD = 0.5

MODEL_NAME = "roberta-base"

# must match your training handle + filename
MODEL_HANDLE = "sharmilamamani/emotion-roberta-aug/pytorch/roberta-bertaug-v1"
MODEL_FILENAME = "roberta_aug_best.pt"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# -------------------------
# 1. Download model from KaggleHub
# -------------------------
print("Downloading RoBERTa+Aug model from KaggleHub...")
model_dir = kagglehub.model_download(MODEL_HANDLE)
model_path = os.path.join(model_dir, MODEL_FILENAME)

print("Model directory:", model_dir)
print("Model path:", model_path)

# -------------------------
# 2. Load Test Data
# -------------------------
test_df = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/test.csv')
print("Test samples:", len(test_df))

# -------------------------
# 3. Tokenizer & Dataset
# -------------------------
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['text'])
        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {key: val.squeeze(0) for key, val in encoding.items()}

test_dataset = EmotionDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -------------------------
# 4. Model Definition (must match training)
# -------------------------
class RobertaForMultiLabel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

model = RobertaForMultiLabel(MODEL_NAME, len(LABELS)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("RoBERTa+Aug model loaded and ready for inference.")

# -------------------------
# 5. Inference on Test Set
# -------------------------
all_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Running inference"):
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)

        logits = model(input_ids, attn)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)

all_probs = np.vstack(all_probs)
preds_bin = (all_probs > THRESHOLD).astype(int)

print("Predictions shape:", preds_bin.shape)

# -------------------------
# 6. Create Submission
# -------------------------
submission = pd.DataFrame(preds_bin, columns=LABELS)
submission.insert(0, "id", test_df["id"])

submission_filename = "submission.csv"
submission.to_csv(submission_filename, index=False)

print(f"Saved submission file: {submission_filename}")
print(submission.head())


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RoBERTa+Aug model loaded and ready for inference.


Running inference:   0%|          | 0/107 [00:00<?, ?it/s]

Predictions shape: (1707, 5)
Saved submission file: submission_07_roberta_aug.csv
   id  anger  fear  joy  sadness  surprise
0   0      1     1    0        0         0
1   1      0     0    0        0         0
2   2      1     1    0        0         0
3   3      0     1    0        0         0
4   4      0     1    0        0         1
